# OmniDocBench 官方评测审核（锁定 commit）

抓取 commit `193627ae…` 的 README、evaluation/、configs/，用于填写 configs/default.yaml 的官方评测命令模板。


In [ ]:
import json, urllib.request
from pathlib import Path

COMMIT = '193627ae9e97d89188468ed1ee3b7a856ff76044'
API = 'https://api.github.com/repos/opendatalab/OmniDocBench'
RAW = 'https://raw.githubusercontent.com/opendatalab/OmniDocBench/' + COMMIT + '/'
out = Path('/kaggle/working/audit_eval')
out.mkdir(parents=True, exist_ok=True)

def gh_json(url):
    req = urllib.request.Request(url, headers={'User-Agent': 'codex-audit'})
    with urllib.request.urlopen(req, timeout=60) as r:
        return json.loads(r.read().decode('utf-8'))

def fetch_raw(rel, name=None):
    req = urllib.request.Request(RAW + rel, headers={'User-Agent': 'codex-audit'})
    with urllib.request.urlopen(req, timeout=60) as r:
        content = r.read().decode('utf-8')
    target = out / (name or rel)
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(content, encoding='utf-8')
    return content

print('helpers ready')


In [ ]:
# 目录列表
for d in ('evaluation', 'configs', ''):
    url = API + '/contents/' + d + '?ref=' + COMMIT
    try:
        listing = gh_json(url)
        names = [(x.get('name'), x.get('type')) for x in listing]
        (out / ('dir_' + (d or 'root') + '.json')).write_text(
            json.dumps(names, ensure_ascii=False, indent=2), encoding='utf-8')
        print(d or 'root', '->', names)
    except Exception as e:
        print(d, 'ERROR', e)


In [ ]:
# 抓取 README 与 evaluation/configs 下全部小文件
import json as _json

files_to_fetch = ['README.md']
for d in ('evaluation', 'configs'):
    p = out / ('dir_' + d + '.json')
    if p.is_file():
        for name, typ in _json.loads(p.read_text(encoding='utf-8')):
            if typ == 'file' and name.endswith(('.py', '.yaml', '.yml', '.md', '.json', '.txt', '.sh')):
                files_to_fetch.append(d + '/' + name)

ok, failed = [], []
for rel in files_to_fetch:
    try:
        fetch_raw(rel)
        ok.append(rel)
    except Exception as e:
        failed.append((rel, str(e)))
print('fetched:', len(ok), '| failed:', failed)


In [ ]:
# 打印 README 中与运行评测相关的行（不凭记忆推断 CLI）
readme = (out / 'README.md').read_text(encoding='utf-8')
lines = readme.splitlines()
keep = []
for i, line in enumerate(lines):
    low = line.lower()
    if any(k in low for k in ('python ', 'pip install', 'bash ', 'eval', 'config', 'md2md', 'end2end', 'usage')):
        keep.append(f'{i+1}: {line}')
(out / 'readme_eval_lines.txt').write_text('\n'.join(keep), encoding='utf-8')
print('\n'.join(keep[:120]))


In [ ]:
# 打印各 eval 脚本的前 60 行（argparse 与入口）
from pathlib import Path
for p in sorted((out / 'evaluation').glob('*.py')):
    print('=' * 30, p.name, '=' * 30)
    lines = p.read_text(encoding='utf-8').splitlines()
    print('\n'.join(lines[:60]))
